# Notebook 1: Customer Profile Construction
**Fridge Hero — BT5153 Group 9**

Builds `user_profile_enriched.csv` from raw interaction history.
LLM inference prioritises users that overlap with the NB3 evaluation set.

**Run order:** NB1 → NB2 → NB3

## 0. Setup & Imports

In [16]:
import pandas as pd
import numpy as np
import ast
import json
import re
from collections import Counter
from tqdm import tqdm
import requests
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = './data/'
OUT_DIR  = './output/'

import os
os.makedirs(OUT_DIR, exist_ok=True)

OLLAMA_URL   = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'qwen2.5:7b'

MIN_INTERACTIONS  = 3
RATING_THRESHOLD  = 4
TOP_N_INGREDIENTS = 15
TOP_N_TAGS        = 10
SAMPLE_SIZE       = 200  # total LLM inference budget
EVAL_SAMPLE_SIZE  = 50   # must match EVAL_SAMPLE in NB3

print('Setup complete.')

Setup complete.


In [2]:
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    print('Ollama running:', [m['name'] for m in r.json().get('models', [])])
except:
    print('Ollama NOT running — start with: ollama serve')

Ollama running: ['qwen2.5:7b', 'llama3.2:latest', 'nomic-embed-text:latest', 'qwen3:4b', 'qwen2.5:3b', 'gpt-oss:120b-cloud']


## 1. Load Raw Data

In [3]:
interactions = pd.read_csv(DATA_DIR + 'RAW_interactions.csv')
recipes      = pd.read_csv(DATA_DIR + 'RAW_recipes.csv')
print('interactions shape:', interactions.shape)
print('recipes shape     :', recipes.shape)
print('interactions columns:', interactions.columns.tolist())
print('recipes columns     :', recipes.columns.tolist())

interactions shape: (1132367, 5)
recipes shape     : (231637, 12)
interactions columns: ['user_id', 'recipe_id', 'date', 'rating', 'review']
recipes columns     : ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']


In [4]:
print('=== interactions sample ===')
display(interactions.head(3))
print('=== recipes sample ===')
display(recipes.head(3))

=== interactions sample ===


,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...


=== recipes sample ===


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13


## 2. Parse List Columns in Recipes

In [5]:
def safe_parse_list(val):
    if isinstance(val, list): return val
    if pd.isna(val): return []
    try: return ast.literal_eval(val)
    except: return []

def parse_nutrition(val):
    keys = ['calories','total_fat_pdv','sugar_pdv','sodium_pdv','protein_pdv','sat_fat_pdv','carbs_pdv']
    lst  = safe_parse_list(val)
    return dict(zip(keys, lst)) if len(lst) == 7 else {k: np.nan for k in keys}

for col in ['ingredients', 'steps', 'tags']:
    recipes[col] = recipes[col].apply(safe_parse_list)

nutrition_df = recipes['nutrition'].apply(parse_nutrition).apply(pd.Series)
recipes = pd.concat([recipes.drop(columns=['nutrition']), nutrition_df], axis=1)
recipes['ingredients_str'] = recipes['ingredients'].apply(lambda x: ' '.join(x).lower())
recipes['tags_str']        = recipes['tags'].apply(lambda x: ' '.join(x).lower())
print('Parsed. recipes shape:', recipes.shape)

Parsed. recipes shape: (231637, 20)


## 3. Filter & Join

In [6]:
liked = interactions[interactions['rating'] >= RATING_THRESHOLD].copy()
print(f'Total interactions : {len(interactions):,}')
print(f'Liked (rating>={RATING_THRESHOLD}): {len(liked):,}')

merged = liked.merge(
    recipes[['id','name','ingredients','tags','ingredients_str','tags_str',
             'minutes','n_steps','calories','protein_pdv','total_fat_pdv',
             'carbs_pdv','sugar_pdv']],
    left_on='recipe_id', right_on='id', how='inner'
)
print(f'Merged shape: {merged.shape}')

Total interactions : 1,132,367
Liked (rating>=4): 1,003,724
Merged shape: (1003724, 18)


## 4. Aggregate per User → Raw Profile

In [7]:
def top_items(series, n=TOP_N_INGREDIENTS):
    all_items = []
    for val in series:
        all_items.extend(val if isinstance(val, list) else [])
    return [item for item, _ in Counter(all_items).most_common(n)]

agg = merged.groupby('user_id').agg(
    n_recipes_rated  = ('recipe_id',    'count'),
    avg_rating_given = ('rating',       'mean'),
    avg_minutes      = ('minutes',      'mean'),
    avg_n_steps      = ('n_steps',      'mean'),
    avg_calories     = ('calories',     'mean'),
    avg_protein_pdv  = ('protein_pdv',  'mean'),
    avg_fat_pdv      = ('total_fat_pdv','mean'),
    avg_carbs_pdv    = ('carbs_pdv',    'mean'),
).reset_index()

top_ingr = merged.groupby('user_id')['ingredients'].apply(
    lambda s: json.dumps(top_items(s, TOP_N_INGREDIENTS))
).reset_index().rename(columns={'ingredients':'top_ingredients'})

top_tag = merged.groupby('user_id')['tags'].apply(
    lambda s: json.dumps(top_items(s, TOP_N_TAGS))
).reset_index().rename(columns={'tags':'preferred_tags'})

merged['is_vegetarian'] = merged['tags_str'].str.contains('vegetarian', na=False).astype(int)
veg_ratio = merged.groupby('user_id')['is_vegetarian'].mean().reset_index()
veg_ratio.columns = ['user_id', 'vegetarian_ratio']

profile = agg.merge(top_ingr, on='user_id').merge(top_tag, on='user_id').merge(veg_ratio, on='user_id')
profile['low_confidence'] = profile['n_recipes_rated'] < MIN_INTERACTIONS
print('Raw profile shape:', profile.shape)
print('Low-confidence users:', profile['low_confidence'].sum())

Raw profile shape: (180310, 13)
Low-confidence users: 146902


## 5. LLM-Inferred Dietary Preferences (Ollama)

In [8]:
def call_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    try:
        r = requests.post(OLLAMA_URL,
                          json={'model': model, 'prompt': prompt,
                                'stream': False, 'options': {'temperature': 0.0}},
                          timeout=60)
        r.raise_for_status()
        return r.json().get('response', '')
    except Exception as e:
        return f'ERROR: {e}'


INFER_PROMPT = """\
You are a culinary analyst. Based ONLY on the recipe list below, infer this user's food preferences.
Return ONLY valid JSON — no explanation, no markdown fences.

Recipes this user liked:
{recipe_names}

Top ingredients they used: {top_ingredients}
Preferred tags: {preferred_tags}

Return JSON with exactly these keys:
{{
  \"avoids_meat\": true/false,
  \"avoids_vegetables\": true/false,
  \"spicy_preference\": \"low\" | \"medium\" | \"high\",
  \"preferred_cuisine\": [list of cuisines, e.g. \"Thai\", \"Italian\"],
  \"dietary_flags\": [list e.g. \"vegetarian\", \"low-fat\", \"halal\"],
  \"sweet_preference\": \"low\" | \"medium\" | \"high\",
  \"confidence\": \"low\" | \"medium\" | \"high\"
}}
"""


def infer_dietary_preferences(row, liked_recipes_map: dict) -> dict:
    default = {
        'avoids_meat': False, 'avoids_vegetables': False,
        'spicy_preference': 'medium', 'preferred_cuisine': [],
        'dietary_flags': [], 'sweet_preference': 'medium', 'confidence': 'low'
    }
    if row['low_confidence']:
        return default
    uid   = row['user_id']
    names = liked_recipes_map.get(uid, [])
    prompt = INFER_PROMPT.format(
        recipe_names    = ', '.join(names[:20]),
        top_ingredients = row['top_ingredients'],
        preferred_tags  = row['preferred_tags']
    )
    raw = call_ollama(prompt)
    try:
        clean = re.sub(r'```[\w]*', '', raw).strip()
        return json.loads(clean)
    except:
        return default

print('LLM inference function ready.')

LLM inference function ready.


In [10]:
# ── Prioritise eval users so LLM judge in NB3 has real profiles ──────────────
# Load test split to identify which users will appear in the NB3 eval loop
test_interactions = pd.read_csv(DATA_DIR + 'interactions_test.csv')
candidate_eval_ids = set(
    test_interactions[test_interactions['rating'] >= RATING_THRESHOLD]['user_id'].unique()
)

confident_users = profile[~profile['low_confidence']].copy()
priority = confident_users[confident_users['user_id'].isin(candidate_eval_ids)]
others   = confident_users[~confident_users['user_id'].isin(candidate_eval_ids)]

priority_for_eval = priority.sample(EVAL_SAMPLE_SIZE, random_state=42)

n_fill        = max(0, SAMPLE_SIZE - EVAL_SAMPLE_SIZE)
others_sample = others.sample(min(n_fill, len(others)), random_state=42)

sample = pd.concat([priority_for_eval, others_sample]).reset_index(drop=True)

eval_user_ids = priority_for_eval['user_id'].tolist()
pd.DataFrame({'user_id': eval_user_ids}).to_csv(OUT_DIR + 'eval_user_ids.csv', index=False)

print(f'Confident users total  : {len(confident_users):,}')
print(f'Priority (in eval pool): {len(priority):,}')
print(f'Priority sampled       : {len(priority_for_eval)}')
print(f'Others sampled         : {len(others_sample)}')
print(f'Total LLM sample       : {len(sample)}')
print(f'Eval user IDs exported : {len(eval_user_ids)}')
print(f'Saved → {OUT_DIR}eval_user_ids.csv')

Confident users total  : 33,408
Priority (in eval pool): 10,330
Priority sampled       : 50
Others sampled         : 150
Total LLM sample       : 200
Eval user IDs exported : 50
Saved → ./output/eval_user_ids.csv


In [11]:
print(f'Priority pool (confident users in test set): {len(priority)}')
print(f'Total confident users: {len(confident_users)}')
print(f'% in test set: {len(priority)/len(confident_users)*100:.1f}%')
print()
print(f'Sample breakdown:')
print(f'  Priority sampled : {len(priority_for_eval)} / {EVAL_SAMPLE_SIZE} target')
print(f'  Others filled    : {len(others_sample)}')
print(f'  Total            : {len(sample)} / {SAMPLE_SIZE} target')
print()
print(f'Eval user IDs (will get LLM judge in NB3): {len(eval_user_ids)}')

Priority pool (confident users in test set): 10330
Total confident users: 33408
% in test set: 30.9%

Sample breakdown:
  Priority sampled : 50 / 50 target
  Others filled    : 150
  Total            : 200 / 200 target

Eval user IDs (will get LLM judge in NB3): 50


In [12]:
liked_recipes_map = (
    merged.groupby('user_id')['name'].apply(list).to_dict()
)

print(f'Running LLM inference on {len(sample)} users...')
dietary_records = []
for _, row in tqdm(sample.iterrows(), total=len(sample)):
    result = infer_dietary_preferences(row, liked_recipes_map)
    result['user_id'] = row['user_id']
    dietary_records.append(result)

dietary_df = pd.DataFrame(dietary_records)
print('Done.')
dietary_df.head(3)

Running LLM inference on 200 users...


100%|██████████| 200/200 [13:26<00:00,  4.03s/it]

Done.


,avoids_meat,avoids_vegetables,spicy_preference,preferred_cuisine,dietary_flags,sweet_preference,confidence,user_id
0,False,False,low,"[Mediterranean, Middle Eastern, Thai]","[vegetarian, low-sodium, halal]",medium,high,1072892
1,False,False,medium,"[Eastern European, American]",[],low,medium,503421
2,False,False,medium,"[Italian, French, Asian]",[],low,high,51878


## 6. Merge & Save Enriched Profile

In [13]:
enriched = profile.merge(dietary_df, on='user_id', how='left')

enriched['avoids_meat']       = enriched['avoids_meat'].fillna(False)
enriched['avoids_vegetables'] = enriched['avoids_vegetables'].fillna(False)
enriched['spicy_preference']  = enriched['spicy_preference'].fillna('medium')
enriched['sweet_preference']  = enriched['sweet_preference'].fillna('medium')
enriched['preferred_cuisine'] = enriched['preferred_cuisine'].apply(lambda x: x if isinstance(x, list) else [])
enriched['dietary_flags']     = enriched['dietary_flags'].apply(lambda x: x if isinstance(x, list) else [])

for col in ['preferred_cuisine', 'dietary_flags']:
    enriched[col] = enriched[col].apply(json.dumps)

out_path = OUT_DIR + 'user_profile_enriched.csv'
enriched.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print('Shape:', enriched.shape)

Saved → ./output/user_profile_enriched.csv
Shape: (180310, 20)


## 7. Sanity Check

In [14]:
print('=== Profile summary ===')
print(enriched[['n_recipes_rated','avg_rating_given','avg_minutes','avg_calories','vegetarian_ratio']].describe().round(2))
print()
print('avoids_meat:', enriched['avoids_meat'].value_counts().to_dict())
print('spicy:', enriched['spicy_preference'].value_counts().to_dict())
print('low_confidence users:', enriched['low_confidence'].sum())

=== Profile summary ===
       n_recipes_rated  avg_rating_given   avg_minutes  avg_calories  \
count        180310.00         180310.00  1.803100e+05     180310.00   
mean              5.57              4.84  1.757356e+04        500.37   
std              53.60              0.33  5.340665e+06        790.01   
min               1.00              4.00  0.000000e+00          0.00   
25%               1.00              4.98  3.000000e+01        219.30   
50%               1.00              5.00  5.000000e+01        360.80   
75%               2.00              5.00  8.000000e+01        555.10   
max            7661.00              5.00  2.147484e+09     101614.70   

       vegetarian_ratio  
count         180310.00  
mean               0.14  
std                0.31  
min                0.00  
25%                0.00  
50%                0.00  
75%                0.00  
max                1.00  

avoids_meat: {False: 180310}
spicy: {'medium': 180185, 'low': 123, 'high': 2}
low_confidence

In [15]:
# Verify eval users got real profiles
eval_check = enriched[enriched['user_id'].isin(eval_user_ids)][[
    'user_id','n_recipes_rated','preferred_cuisine','spicy_preference','avoids_meat','confidence'
]]
print(f'Eval users with LLM profile: {len(eval_check)}')
print(eval_check.to_string())

Eval users with LLM profile: 50
       user_id  n_recipes_rated                                 preferred_cuisine spicy_preference avoids_meat confidence
331       6536                7                           ["Tex-Mex", "American"]           medium       False       high
400       7676               18      ["American", "German", "Indian", "Moroccan"]           medium       False       high
3477     37947               14                             ["Mexican", "Indian"]           medium       False       high
3856     40497                4                           ["Chinese", "American"]              low       False     medium
3946     41201                5                           ["Italian", "American"]              low       False     medium
4091     42058               20     ["American", "Southern", "French", "Italian"]           medium       False       high
5221     51878               10                    ["Italian", "French", "Asian"]           medium       False    